# 04 · Regime shift detection

Cluster the latent cloud into regimes, build a transition matrix, and compute the critical-transition warning score along one participant's trajectory.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt


In [ ]:
from data.synthetic_generator import generate_synthetic_cohort
from features import engineer_all_features
from states.latent_state_encoder import encode_latent_states_classical
from states.regime_detector import fit_regime_detector, transition_matrix
from states.early_warning import critical_transition_warning_score

df = engineer_all_features(generate_synthetic_cohort(n_participants=60, n_days=120, seed=17))


In [ ]:
def pick(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].to_numpy(dtype=float, na_value=0.0) if present else np.zeros((len(df), len(cols)))

W = pick(df, ['sleep_duration_hours','hrv_rmssd','resting_hr','daily_steps','recovery_score'])
B = pick(df, ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count'])
C = pick(df, ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag'])
M = pick(df, ['missing_wearable_flag','missing_phone_flag','missing_survey_flag'])
P = pick(df, ['baseline_hrv','baseline_resting_hr','baseline_climate_vulnerability','baseline_resilience'])
Z = encode_latent_states_classical(W, B, C, M, P).latent
detector = fit_regime_detector(Z, n_clusters=4, random_state=17)
labels = detector.predict(Z)
pd.Series(labels).value_counts()


## Transition matrix


In [ ]:
T, regimes = transition_matrix(labels)
pd.DataFrame(T, index=regimes, columns=regimes).round(3)


## Critical-transition warning score


In [ ]:
pid = df['participant_id'].iloc[0]
mask = df['participant_id'].values == pid
pZ = Z[mask]
w = critical_transition_warning_score(pZ)
print({k: (np.nanmean(v) if hasattr(v,'__len__') else v) for k,v in w.items() if v is not None})
